# Repeat Purchase Validation

**Why this notebook:** **Repeat purchases drive CLV.** If `retention_score` accumulation or decay is wrong, forward revenue and churn-adjusted valuations are wrong — even when the single-day purchase formula is correct.

**Audience:** Engineers editing `register_purchase` / `decay_retention` / repeat logic.

**Outcome:** Multi-day trajectories, ordering checks across `repeat_boost`, and proof the **budget gate** dominates loyalty.

**Regression test and visual reference for multi-day retention mechanics in `app/domain/customer.py`.**

This notebook validates how a customer's `retention_score` accumulates after purchases and decays when time passes — and how that score feeds back into the daily purchase probability. Run it after any change to `register_purchase`, `decay_retention`, or the repeat-purchase logic.

| Section | What is validated | Needs DB? |
|---------|-------------------|-----------|
| §0 | Setup & imports | No |
| §1 | Retention score mechanics — formula reference | No |
| §2 | Accumulation — repeated purchases | No |
| §3 | Decay — time without purchase | No |
| §4 | Accumulate then decay — realistic trajectory | No |
| §5 | Retention score → purchase probability feedback | No |
| §6 | Multi-customer comparison — different repeat_boost levels | No |
| §7 | Budget constraint persists despite high retention | No |

**No database required.** Run from the repo root with the venv active and `pip install -e ".[dev]"`.


## §0  Setup & imports


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from app.domain.customer import Customer, PurchaseContext

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.35

RNG_SEED = 42
BASE_CTX = PurchaseContext(temporal_multiplier=1.0, geographic_multiplier=1.0, promo_eligible=True)

def make_customer(**overrides) -> Customer:
    """Return a fresh Customer from a canonical baseline, applying any overrides."""
    defaults = dict(
        customer_id=1,
        customer_index=0,
        budget=80.0,
        buy_propensity=0.25,
        price_threshold=35.0,
        repeat_boost=0.35,
        basket_mean=20.0,
        location_zone="A",
        retention_sensitivity=0.5,
        promo_sensitivity=0.5,
    )
    defaults.update(overrides)
    return Customer(**defaults)

print("Setup complete — no database required.")


### Follow-up questions

- Where do `register_purchase` and `decay_retention` live in the production customer object?
- Why validate retention in isolation before trusting CLV and churn simulations?
- If imports fail, is the fix usually path-related or package-related?



## §1  Retention score mechanics — formula reference

`retention_score` is a continuous loyalty signal that lives on the `Customer` object. It starts at `1.0` (neutral), increases after purchases, and decays when the customer goes quiet.

### Accumulation — `register_purchase(day, order_value)`

After each successful purchase, the score is bumped:

$$\text{retention\_score} \leftarrow \min\!\left(2.5,\; \text{retention\_score} + 0.12 \times \text{retention\_sensitivity}\right)$$

- At `retention_sensitivity = 0.5` (default): each purchase adds **+0.06**
- Hard ceiling at **2.5** — a highly loyal customer saturates after ~25 orders

### Decay — `decay_retention(days_since_purchase)`

Called once per day before the purchase decision; subtracts:

$$\text{retention\_score} \leftarrow \max\!\left(1.0,\; \text{retention\_score} - 0.01 \times \min(d, 30)\right)$$

where `d = days_since_purchase`.

- Gaps of 1 day → decay of **0.01**; gaps of 30+ days → maximum single-step decay of **0.30**
- Hard floor at **1.0** — retention score never goes below neutral

### Effect on purchase probability

The score feeds into `compute_purchase_probability` via:

$$\text{retention\_factor} = 1 + 0.15 \times (\text{score} - 1) \times \text{retention\_sensitivity}$$

This is validated in §5.


### Follow-up questions

- In words, what is the retention score ceiling and floor, and how does churn hazard use them?
- Which formula line would you change to make loyalty decay faster after a long gap?
- How does retention feed back into purchase probability in `customer_model_validation.ipynb`?



## §2  Accumulation — repeated purchases

Each call to `register_purchase` should raise `retention_score` by exactly `0.12 × retention_sensitivity`, capping at 2.5.


In [ ]:
N_PURCHASES = 30
sensitivity = 0.5
expected_bump = 0.12 * sensitivity  # 0.06 per purchase at default sensitivity

c = make_customer(retention_sensitivity=sensitivity)
score_history = [c.retention_score]

for day in range(1, N_PURCHASES + 1):
    c.register_purchase(day, order_value=20.0)
    score_history.append(c.retention_score)

score_history = np.array(score_history)

# ── Assertions ────────────────────────────────────────────────────────────
# Before ceiling: each step must increase by exactly expected_bump
for i, (s_before, s_after) in enumerate(zip(score_history[:-1], score_history[1:])):
    if s_before < 2.5:  # only assert before ceiling is hit
        expected = min(2.5, s_before + expected_bump)
        assert abs(s_after - expected) < 1e-9, (
            f"purchase {i+1}: expected score {expected:.4f}, got {s_after:.4f}"
        )

# Score must never exceed ceiling
assert score_history.max() <= 2.5 + 1e-9, "retention_score exceeded ceiling of 2.5"
# Score must never fall below floor
assert score_history.min() >= 1.0 - 1e-9, "retention_score fell below floor of 1.0"
# Final score should be at ceiling (30 purchases × 0.06 = 1.8 → floor 1.0 + 1.8 = 2.8 → capped)
assert abs(score_history[-1] - 2.5) < 1e-9, "score should saturate at 2.5 after many purchases"

purchases_to_saturate = int(np.ceil((2.5 - 1.0) / expected_bump))
print(f"Bump per purchase (sensitivity={sensitivity}): +{expected_bump:.4f}")
print(f"Purchases to reach ceiling 2.5: {purchases_to_saturate}")
print(f"Final score after {N_PURCHASES} purchases: {score_history[-1]:.4f}")
print("All accumulation assertions passed")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(N_PURCHASES + 1), score_history, lw=2, color="#5BA4CF", marker="o", ms=4, label="retention_score")
ax.axhline(2.5, color="crimson", lw=1.5, ls="--", label="ceiling = 2.5")
ax.axhline(1.0, color="gray",   lw=1.0, ls=":",  label="floor = 1.0")
ax.axvline(purchases_to_saturate, color="darkorange", lw=1.2, ls="--",
           label=f"saturation at purchase #{purchases_to_saturate}")
ax.set_xlabel("Number of purchases")
ax.set_ylabel("retention_score")
ax.set_title(f"§2  Accumulation — +{expected_bump:.2f} per purchase (sensitivity={sensitivity})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### Follow-up questions

- Plot or list score after each purchase — is growth linear, concave, or capped?
- What happens if `register_purchase` is called twice on the same simulated day?
- Add an assertion that score never exceeds the documented maximum.



## §3  Decay — time without purchase

After purchases stop, `decay_retention(days_since)` erodes the score back toward the floor. The decay amount per call is `0.01 × min(days_since, 30)` — so a 1-day gap costs 0.01 and a 30-day gap costs 0.30.


In [ ]:
# Start from a highly loyal customer (score near ceiling)
c = make_customer(retention_score=2.5)
gaps = list(range(1, 41))  # day-gaps 1..40

# Test each gap independently (not cumulative)
scores_after_gap = []
expected_scores  = []
for gap in gaps:
    c_tmp = make_customer(retention_score=2.5)
    c_tmp.decay_retention(gap)
    scores_after_gap.append(c_tmp.retention_score)
    expected = max(1.0, 2.5 - 0.01 * min(gap, 30))
    expected_scores.append(expected)

# ── Assertions ────────────────────────────────────────────────────────────
for gap, actual, expected in zip(gaps, scores_after_gap, expected_scores):
    assert abs(actual - expected) < 1e-9, f"gap={gap}: expected {expected:.4f}, got {actual:.4f}"

# Score must never go below floor
assert min(scores_after_gap) >= 1.0 - 1e-9, "decay_retention violated floor of 1.0"

# At gap >= 15 days from 2.5: score = 2.5 - 0.15 = 2.35
idx_15 = gaps.index(15)
assert abs(scores_after_gap[idx_15] - 2.35) < 1e-6, "15-day gap from 2.5 should yield 2.35"

# At gap >= 30 days from 2.5: floor applied if needed — 2.5 - 0.30 = 2.20
idx_30 = gaps.index(30)
assert abs(scores_after_gap[idx_30] - 2.20) < 1e-6, "30-day gap from 2.5 should yield 2.20"

# Beyond 30 days: decay is capped at min(d, 30) → same as 30-day gap
assert scores_after_gap[39] == scores_after_gap[29], "decay beyond 30 days should be identical to 30-day decay"

print("All decay assertions passed")
print(f"Score after 1-day gap:  {scores_after_gap[0]:.4f}  (expected {expected_scores[0]:.4f})")
print(f"Score after 15-day gap: {scores_after_gap[14]:.4f}  (expected {expected_scores[14]:.4f})")
print(f"Score after 30-day gap: {scores_after_gap[29]:.4f}  (expected {expected_scores[29]:.4f})")
print(f"Score after 40-day gap: {scores_after_gap[39]:.4f}  (expected {expected_scores[39]:.4f}, same as 30-day)")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gaps, scores_after_gap, lw=2, color="#E07A5F", marker="o", ms=4, label="score after single gap")
ax.axhline(1.0, color="gray",   lw=1, ls=":",  label="floor = 1.0")
ax.axhline(2.5, color="crimson",lw=1, ls="--", label="starting score = 2.5")
ax.axvline(30,  color="steelblue", lw=1.2, ls="--", label="cap at 30 days")
ax.annotate("decay capped\n(> 30 days = same)",
            xy=(30, scores_after_gap[29]), xytext=(33, scores_after_gap[29] + 0.08),
            fontsize=8, color="steelblue",
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=1))
ax.set_xlabel("days_since_purchase argument")
ax.set_ylabel("retention_score after decay")
ax.set_title("§3  Decay — single call to decay_retention from score = 2.5")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### Follow-up questions

- After long decay, how close does score get to the floor — analytically vs empirically?
- If decay rate changed, which simulator outputs (churn, CLV) would move first?
- Compare decay-only trajectory to the combined trajectory in §4 — when do they diverge?



## §4  Accumulate then decay — realistic multi-day trajectory

In the live engine, `decay_retention` and `decide_purchase`/`register_purchase` interleave on every simulated day. This section runs a realistic 60-day scenario with sporadic purchases and tracks the `retention_score` day-by-day.

We force purchases on specific days to get a deterministic trajectory (bypassing the Bernoulli draw) so the trajectory is reproducible regardless of RNG state.


In [ ]:
HORIZON = 60
# Force purchase on these days to create a realistic pattern of activity and quiet
PURCHASE_DAYS = {3, 7, 8, 12, 20, 21, 22, 35, 50, 55, 56, 57}

c = make_customer(retention_sensitivity=0.5)
score_by_day = []   # score at start of each day (before decay)
purchased    = []   # boolean: did a purchase happen?

for day in range(1, HORIZON + 1):
    # Decay first (engine order: decay → decide → register)
    if c.last_purchase_day is not None:
        c.decay_retention(day - c.last_purchase_day)

    score_by_day.append(c.retention_score)

    bought = day in PURCHASE_DAYS
    if bought:
        c.register_purchase(day, order_value=20.0)
    purchased.append(bought)

score_by_day = np.array(score_by_day)

# ── Assertions ────────────────────────────────────────────────────────────
assert score_by_day.max() <= 2.5 + 1e-9, "score exceeded ceiling"
assert score_by_day.min() >= 1.0 - 1e-9, "score fell below floor"

# After a cluster of purchases score should be higher than after a long quiet period
score_after_cluster = score_by_day[22]   # day 23 — just after days 20/21/22
score_after_gap     = score_by_day[34]   # day 35 — 13 days after last purchase
assert score_after_cluster > score_after_gap, "score should be higher after recent purchases"
print(f"Score day 23 (after purchase cluster): {score_after_cluster:.4f}")
print(f"Score day 35 (13-day gap):             {score_after_gap:.4f}")
print("Trajectory assertions passed")

# ── Plot ──────────────────────────────────────────────────────────────────
days = np.arange(1, HORIZON + 1)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(days, score_by_day, lw=2, color="#5BA4CF", zorder=3, label="retention_score")
ax.fill_between(days, 1.0, score_by_day, alpha=0.15, color="#5BA4CF")

# Mark purchases as vertical lines
purchase_days_sorted = sorted(PURCHASE_DAYS)
for i, pd_day in enumerate(purchase_days_sorted):
    ax.axvline(pd_day, color="#E07A5F", lw=1.5, alpha=0.7,
               label="purchase" if i == 0 else None)

ax.axhline(2.5, color="crimson", lw=1, ls="--", label="ceiling = 2.5")
ax.axhline(1.0, color="gray",   lw=1, ls=":",  label="floor = 1.0")

ax.set_xlabel("Simulated day")
ax.set_ylabel("retention_score")
ax.set_title("§4  Accumulate then decay — 60-day retention trajectory\n"
             "(vertical lines = purchase events)")
ax.set_xlim(1, HORIZON)
ax.set_ylim(0.9, 2.6)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### Follow-up questions

- Does the realistic trajectory show plausible “bursts” of purchases — if not, what parameter would you tune?
- How many days of inactivity does it take to wipe out most loyalty gains?
- Export this trajectory to CSV — what columns would a PM want to see?



## §5  Retention score → purchase probability feedback

The `retention_factor` in the probability formula:

$$\text{retention\_factor} = 1 + 0.15 \times (\text{score} - 1) \times \text{retention\_sensitivity}$$

means a customer who has been buying regularly (high score) has a meaningfully higher daily purchase probability than an identically parameterised customer who lapsed (score decayed to floor). Here we make this concrete by comparing two customers with identical parameters but different purchase histories.


In [ ]:
price = 25.0

# Loyal customer: 10 purchases, score bumped accordingly
c_loyal = make_customer(has_purchased_before=True, purchase_count=10, retention_sensitivity=0.5)
for i in range(10):
    c_loyal.register_purchase(i + 1, 20.0)

# Lapsed customer: same parameters, no recent activity → score at floor
c_lapsed = make_customer(has_purchased_before=False, purchase_count=0, retention_sensitivity=0.5)
# score is already 1.0 (floor)

p_loyal  = c_loyal.compute_purchase_probability(price, 1, BASE_CTX)
p_lapsed = c_lapsed.compute_purchase_probability(price, 1, BASE_CTX)

print(f"Loyal customer — retention_score: {c_loyal.retention_score:.4f},  p: {p_loyal:.4f}")
print(f"Lapsed customer — retention_score: {c_lapsed.retention_score:.4f}, p: {p_lapsed:.4f}")
print(f"Absolute lift from retention: {p_loyal - p_lapsed:.4f}")

# ── Assertion ─────────────────────────────────────────────────────────────
assert p_loyal > p_lapsed, "loyal customer must have higher p than lapsed customer"
print("Assertion passed: loyal p > lapsed p")

# ── Sweep: show how p evolves as retention_score climbs through buy history ─
n_purchases_range = range(0, 26)
probs_by_purchases = []
scores_by_purchases = []

for n in n_purchases_range:
    c = make_customer(has_purchased_before=(n > 0), purchase_count=n, retention_sensitivity=0.5)
    for i in range(n):
        c.register_purchase(i + 1, 20.0)
    probs_by_purchases.append(c.compute_purchase_probability(price, 1, BASE_CTX))
    scores_by_purchases.append(c.retention_score)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.bar(["Lapsed\n(0 orders,\nscore=1.0)", "Loyal\n(10 orders,\nscore=2.5)"],
       [p_lapsed, p_loyal], color=["#5BA4CF", "#E07A5F"], edgecolor="white", width=0.45)
for x, y, label in zip([0, 1], [p_lapsed, p_loyal], [p_lapsed, p_loyal]):
    ax.text(x, y + 0.005, f"{label:.3f}", ha="center", fontsize=10)
ax.set_ylabel("Purchase probability")
ax.set_title("Head-to-head at same base params")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, max(p_loyal, p_lapsed) * 1.35)

ax = axes[1]
color1 = "#5BA4CF"
color2 = "#E07A5F"
l1, = ax.plot(n_purchases_range, probs_by_purchases, lw=2, color=color1, label="purchase probability")
ax2 = ax.twinx()
l2, = ax2.plot(n_purchases_range, scores_by_purchases, lw=2, color=color2, ls="--", label="retention_score")
ax2.set_ylabel("retention_score", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)
ax2.set_ylim(0.9, 2.7)
ax.set_xlabel("Cumulative purchases")
ax.set_ylabel("Purchase probability", color=color1)
ax.tick_params(axis="y", labelcolor=color1)
ax.set_title("p and retention_score vs order count")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
lines = [l1, l2]
ax.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="lower right")

fig.suptitle("§5  Retention score → purchase probability feedback", fontsize=11)
plt.tight_layout()
plt.show()


### Follow-up questions

- When retention is high, does purchase probability plateau — is that desirable?
- Couple this with budget: can high retention still yield zero purchases?
- Sketch a test that fails if feedback from purchases to probability is accidentally disabled.



## §6  Multi-customer comparison — different `repeat_boost` levels

`repeat_boost` controls how much a returning customer's purchase count amplifies their probability. Customers with higher `repeat_boost` become repeat buyers more aggressively, so their cumulative purchase count over a fixed horizon should be ordered by boost level.

We run three customers through an identical 30-day loop under the same RNG seed so the only difference is their `repeat_boost`.

**Performance:** This cell averages **500 trials** per boost level — it is the slowest section. `%%time` reports wall time. Some CI configs set `SKIP_NOTEBOOK_TESTS=1` to skip notebook execution entirely (see `notebooks/README.md`).


In [ ]:
%%time
# Heavy Monte Carlo (500 seeds × 3 boost levels). For full-suite CI, see SKIP_NOTEBOOK_TESTS in notebooks/README.md.
HORIZON = 30
PRICE   = 28.0
BOOST_LEVELS = [0.10, 0.35, 0.60]
N_TRIALS = 500  # independent runs per boost level to average out noise

def simulate_customer(boost: float, seed: int) -> dict:
    """Run one customer through HORIZON days; return day-by-day state."""
    rng = np.random.default_rng(seed)
    c = make_customer(repeat_boost=boost, buy_propensity=0.25)
    cumulative_purchases = []
    scores = []
    for day in range(1, HORIZON + 1):
        if c.last_purchase_day is not None:
            c.decay_retention(day - c.last_purchase_day)
        scores.append(c.retention_score)
        if c.decide_purchase(rng, PRICE, day, BASE_CTX):
            c.register_purchase(day, PRICE)
        cumulative_purchases.append(c.purchase_count)
    return {"purchases": cumulative_purchases, "scores": scores, "total": c.purchase_count}

# Average over N_TRIALS seeds
avg_cumulative = {}
total_means = {}
for boost in BOOST_LEVELS:
    all_runs = np.array([simulate_customer(boost, seed)["purchases"] for seed in range(N_TRIALS)])
    avg_cumulative[boost] = all_runs.mean(axis=0)
    total_means[boost] = all_runs[:, -1].mean()

# ── Assertions ────────────────────────────────────────────────────────────
sorted_boosts = sorted(BOOST_LEVELS)
for i in range(1, len(sorted_boosts)):
    b_low, b_high = sorted_boosts[i-1], sorted_boosts[i]
    assert total_means[b_high] >= total_means[b_low], (
        f"boost={b_high} should yield >= purchases than boost={b_low}: "
        f"{total_means[b_high]:.2f} vs {total_means[b_low]:.2f}"
    )
print("Purchase count ordering follows boost level: PASSED")
for b in BOOST_LEVELS:
    print(f"  repeat_boost={b}: avg {total_means[b]:.2f} purchases over {HORIZON} days")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
colors = ["#5BA4CF", "#E07A5F", "#3D405B"]

ax = axes[0]
for boost, color in zip(BOOST_LEVELS, colors):
    ax.plot(range(1, HORIZON + 1), avg_cumulative[boost], lw=2, color=color,
            label=f"repeat_boost = {boost}")
ax.set_xlabel("Simulated day")
ax.set_ylabel("Avg cumulative purchases")
ax.set_title(f"Cumulative purchases (avg over {N_TRIALS} runs)")
ax.legend(fontsize=9)

ax = axes[1]
bars = ax.bar([str(b) for b in BOOST_LEVELS], [total_means[b] for b in BOOST_LEVELS],
              color=colors, edgecolor="white")
for bar, b in zip(bars, BOOST_LEVELS):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{total_means[b]:.2f}", ha="center", fontsize=10)
ax.set_xlabel("repeat_boost")
ax.set_ylabel(f"Avg purchases over {HORIZON} days")
ax.set_title("Final tally by boost level")

fig.suptitle("§6  Multi-customer comparison — repeat_boost drives purchase frequency", fontsize=11)
plt.tight_layout()
plt.show()


### Follow-up questions

- Which `repeat_boost` level produces the highest end-of-horizon purchase rate — is it monotonic?
- How would you explain heterogeneity across synthetic customers without overfitting a story?
- If two customers differ only by `repeat_boost`, where else in the engine must stay identical?



## §7  Budget constraint persists despite high retention

No matter how loyal a customer becomes (high `retention_score`, many past purchases, high `repeat_boost`), if the offered price exceeds their `budget`, `compute_purchase_probability` returns `0.0` and `decide_purchase` never returns `True`. The budget gate is unconditional.

We verify this with 1 000 Bernoulli trials for a fully saturated, highly boosted customer presented with a price that exceeds their budget.


In [ ]:
rng = np.random.default_rng(0)
N_TRIALS = 1_000

# Build the most "willing" customer imaginable
c_max = make_customer(
    budget=40.0,
    buy_propensity=0.99,
    repeat_boost=0.99,
    has_purchased_before=True,
    purchase_count=100,
    retention_sensitivity=1.0,
    retention_score=2.5,
)

price_below_budget = 39.99
price_above_budget = 40.01

p_below = c_max.compute_purchase_probability(price_below_budget, 1, BASE_CTX)
p_above = c_max.compute_purchase_probability(price_above_budget, 1, BASE_CTX)

# ── Probability assertions ─────────────────────────────────────────────────
assert p_above == 0.0, f"p must be exactly 0 above budget; got {p_above}"
assert p_below > 0.0, f"p must be > 0 below budget; got {p_below}"
print(f"p at ${price_below_budget} (below budget ${c_max.budget}): {p_below:.4f}")
print(f"p at ${price_above_budget} (above budget ${c_max.budget}): {p_above:.4f}")

# ── Bernoulli trials above budget: must be zero purchases ─────────────────
buys_above = sum(c_max.decide_purchase(rng, price_above_budget, 1, BASE_CTX) for _ in range(N_TRIALS))
assert buys_above == 0, f"Expected 0 purchases above budget, got {buys_above}/{N_TRIALS}"

# ── Bernoulli trials below budget: must be non-zero ───────────────────────
buys_below = sum(c_max.decide_purchase(rng, price_below_budget, 1, BASE_CTX) for _ in range(N_TRIALS))
assert buys_below > 0, f"Expected > 0 purchases below budget, got {buys_below}/{N_TRIALS}"

print(f"\nBernoulli draws at ${price_above_budget}: {buys_above}/{N_TRIALS} purchases (expected 0)")
print(f"Bernoulli draws at ${price_below_budget}: {buys_below}/{N_TRIALS} purchases (empirical {buys_below/N_TRIALS:.3f} ≈ theoretical {p_below:.3f})")
print("\nAll budget constraint assertions passed")

# ── Plot ──────────────────────────────────────────────────────────────────
prices_sweep = np.linspace(20.0, 55.0, 400)
probs_sweep  = [c_max.compute_purchase_probability(float(p), 1, BASE_CTX) for p in prices_sweep]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(prices_sweep, probs_sweep, lw=2.5, color="#5BA4CF", label="purchase probability")
ax.axvline(c_max.budget, color="crimson", lw=2, ls="--",
           label=f"budget = ${c_max.budget} (hard cutoff)")
ax.fill_between(prices_sweep, 0, probs_sweep,
                where=[p > c_max.budget for p in prices_sweep],
                alpha=0.15, color="crimson", label="p = 0 above budget")
ax.annotate(f"{buys_above}/{N_TRIALS}\npurchases",
            xy=(45, 0.02), fontsize=10, color="crimson", ha="center")
ax.set_xlabel("Offered price ($)")
ax.set_ylabel("Purchase probability")
ax.set_title("§7  Budget constraint — loyal, high-retention customer\n"
             "budget is unconditional regardless of retention or boost")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### Business read — retention vs predictive CLV

Higher **`retention_score`** (from recent purchases) increases **`compute_purchase_probability`**, which flows into **`compute_predictive_clv`** via higher expected daily revenue. The snippet below holds basket and price fixed so only retention differs — mirroring why §12 in `03_ab_and_clv.ipynb` matters end-to-end.


In [ ]:
_offer = 26.0
_day = 15
_churn = 0.002
for label, rs in [("Lower retention", 1.05), ("Higher retention", 1.75)]:
    cx = make_customer(
        buy_propensity=0.28,
        retention_score=rs,
        has_purchased_before=True,
        purchase_count=2,
    )
    clv = cx.compute_predictive_clv(90, 0.10, _churn, _offer, _day, BASE_CTX)
    p = cx.compute_purchase_probability(_offer, _day, BASE_CTX)
    print(f"{label}: retention_score={rs:.2f}  p_buy={p:.3f}  predictive_CLV_90d=${clv:.2f}")


### Follow-up questions

- Show a case where retention is strong but spend is capped — what business narrative fits?
- Why might CLV still be finite even when loyalty metrics look excellent?
- Link this section to holdout validation in `03_ab_and_clv.ipynb`.



## Summary

All assertions are inline and raise `AssertionError` on regression. The table below summarises each section:

| Section | Method under test | Assertion |
|---------|-------------------|-----------|
| §2 | `register_purchase` | Each call adds exactly `0.12 × sensitivity`; ceiling 2.5 enforced |
| §3 | `decay_retention` | Single call subtracts `0.01 × min(d, 30)`; floor 1.0 enforced; cap at 30 days verified |
| §4 | Both together | Score never exceeds ceiling or falls below floor over 60-day realistic trajectory |
| §5 | Both → `compute_purchase_probability` | Loyal customer (high score) has strictly higher `p` than lapsed customer |
| §6 | Full loop | Average purchase count ordered by `repeat_boost` (N=500 runs per level) |
| §7 | `decide_purchase` + budget gate | 0 purchases above budget across 1 000 trials; >0 below budget |


### What you learned · Next up

- Retention dynamics are **asserted** across accumulation, decay, boost ordering, and the **hard budget** rule.
- **Next:** [`03_ab_and_clv.ipynb`](03_ab_and_clv.ipynb) §12 — calibrate **predictive CLV** vs **holdout actuals** at run scale.

